### Indexing Pipeline
#### Step 1: Data Loading

In [1]:
import requests 
from bs4 import BeautifulSoup # import BeautifulSoup for parsing HTML content

# ------------------------------------------------------------
# 1. Fetch HTML Content from Wikipedia API
# ------------------------------------------------------------

url = "https://en.wikipedia.org/w/api.php" # Wikipedia API endpoint

headers = { # headers for the API request
    "User-Agent": "my-rag-project/1.0 (contact: your-email@example.com)"  # Replace with your contact information
}

params = { # parameters for the API request
    
    "action": "parse", # action to parse the page

    "page": "Retrieval-augmented_generation", # page title to fetch

    "prop": "text", # property to get the HTML content

    "format": "json", # response format

    "formatversion": "2"   # version of the format
}

response = requests.get( # make the API request
    
    url,   # API endpoint

    params=params, # parameters for the request

    headers=headers # headers for the request
)

response.raise_for_status() # raise an error if the request was unsuccessful

html = response.json()["parse"]["text"] # extract the HTML content from the JSON response

# print("Original HTML:", len(html))  # print the length of the original HTML content

# ------------------------------------------------------------
# 2. Remove MediaWiki Parser Report
# ------------------------------------------------------------

parser_start = html.find("NewPP limit report") # find the starting index of the parser report in the HTML content

if parser_start != -1: # if the parser report is found, truncate the HTML content to remove it

    html = html[:parser_start] # truncate the HTML before the parser report

# ------------------------------------------------------------
# 3. Remove Unwanted Elements
# ------------------------------------------------------------

soup = BeautifulSoup(html, "html.parser") # parse the HTML content into a BeautifulSoup object

selectors = [
    "style", # select style elements
    "script", # select script elements
    ".mw-editsection", # select edit section elements
    ".reference",  # select reference elements
    ".reflist",  # select reference list elements
    ".navbox", # select navigation box elements
    ".metadata", # select metadata elements
    ".infobox", # select infobox elements
    ".toc",  # select table of contents elements
    "#toc",  # select table of contents elements with id "toc"
]

for selector in selectors: # iterate through each selector

    for element in soup.select(selector):
        
        element.decompose()  # remove the element from the soup

# ------------------------------------------------------------
# 4. Remove References Heading
# ------------------------------------------------------------

for heading in soup.find_all(["h2", "h3"]):  # find all h2 and h3 headings in the soup

    if heading.get_text(" ", strip=True).lower() == "references": # if the heading text is "references" (case-insensitive)

        heading.decompose() # remove the heading from the soup

        break # break the loop after removing the references heading

# ------------------------------------------------------------
# 5. Convert Cleaned Soup Back to HTML String
# ------------------------------------------------------------

article_html = str(soup) # convert the cleaned soup back to an HTML string

# print("Cleaned HTML:", len(article_html)) # print the length of the cleaned HTML content

# print("\nFirst 1000 characters:") 
# print(article_html[:1000]) # print the first 1000 characters of the cleaned HTML content

### Step 2: Part 1: Structure-aware Chunking

In [2]:
from langchain_text_splitters import HTMLSectionSplitter

html_splitter = HTMLSectionSplitter(
    headers_to_split_on=[ # list of headers to split on
        
        ("h1", "Header 1"),
        ("h2", "Header 2"),
        ("h3", "Header 3"),
        ("h4", "Header 4"),
        ("h5", "Header 5"),
        ("h6", "Header 6"),
    ]
)

sections = html_splitter.split_text(article_html) # split the cleaned HTML content into sections based on the specified headers

# print("Number of sections:", len(sections)) 
# print(sections)

### Step 2: Part 2: Fixed-size chunking:

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(  # Initialize the text splitter
    chunk_size=1000,  # Set the maximum number of characters in each chunk
    chunk_overlap=200 # Set the number of characters to overlap between chunks
)

chunks = text_splitter.split_documents(sections)   # Split the sections into smaller chunks based on the specified chunk size and overlap

# print("Number of chunks:", len(chunks))

# for i, chunk in enumerate(chunks): # iterate through each chunk and print its details
#     print(f"\n{'='*80}")
#     print(f"CHUNK {i+1}")
#     print(f"Characters: {len(chunk.page_content)}")
#     print(f"Metadata: {chunk.metadata}")
#     print(f"{'='*80}")
#     print(chunk.page_content)